# 43008: Reinforcement Learning - WOW Naval MAPPO with a Fleet Commander

## Stage 1: Capture

[Imitation](StageP-Imitation.ipynb) · [Stage 0](Stage0-Gunnery.ipynb) · **Stage 1** · [Stage 2](Stage2-Defend.ipynb) · [Stage 3](Stage3-Duel.ipynb) · [Stage 4](Stage4-King-of-the-Hill.ipynb) · [Stage 5](Stage5-Archipelago.ipynb) · [Stage 6](Stage6-Domination.ipynb) · [Stage 7](Stage7-Open.ipynb) · [the full walkthrough](WOW-MAPPO-Simple.ipynb)

| Stage | Battle | Fleet | Opponent | Promotion requirement |
|---|---|---|---|---|
| 1 - Capture | `capture` | 1 cruiser | Passive destroyer, far away | circle captured in ≥70% of the last 50 battles |

### What the fleet learns here

**Capture the circle.** One cruiser, one neutral capture circle 4.5 km ahead; the only enemy is a passive destroyer 16 km beyond it, out of sight. Points come only from holding the circle, so the cruiser has to sail into it and stay inside until it is captured.

Measured: random play captures the circle in 0% of battles, the script "sail to circle A" in 100%.

### One model through every stage

All stage notebooks train the **same model with the same method**. This notebook continues the run's latest checkpoint, which carries everything learned so far - the networks, the optimisers, the value normalisers, the league, the timestep count and the stage - and the run's battle log, from which the promotion window continues where the last session stopped.

* **Training is continuous.** Not passed yet? **Run this notebook again** - each session continues the last one (yours or a teammate's), from `models/model_latest.pt`, which is refreshed every `save_interval` (so even a crashed session loses little). `KEEP_GOING = True` (configuration cell) carries on into the next stages in the same session instead of stopping at the promotion.
* Passed? Training stops and saves, and the model moves to stage 2: open [Stage 2](Stage2-Defend.ipynb).


### The method in one picture

| Piece | What it does | Where it comes from | Course link |
|---|---|---|---|
| **Captain** (every ship, one shared network) | its own view → 6 action heads; a transformer over ships/circles/islands, a GRU memory, pointer heads for *which target* and *which circle* | AlphaStar, OpenAI Five | W10-C custom feature extractor; frame stacking → memory |
| **Fleet commander** | every 10 s: an order per ship - free / engage / hold circle k | Honor of Kings' hierarchical macro strategy | the same PPO at a slower clock |
| **Centralised critic** | sees the true battle (training only); one value per reward group + win probability | MAPPO, Honor of Kings multi-head value | W10-A/B actor-critic |
| **Update** | PPO with dual clip, value normalisation and clipping | MAPPO, Honor of Kings | W10-B PPO |
| **Warm start** | the captain first copies the Elite rule AI (StageP), then a fading KL penalty keeps it close | AlphaStar | W9-A supervised training loop |
| **League** | the enemy is the rule AI, our latest copy, or a past copy (PFSP) - with Elo ratings | OpenAI Five, AlphaStar | W9-C/D frozen target network |
| **Curriculum** | 8 stages, each passed at ≥70% of its own goal; earlier stages are replayed 10-15% of the time | TiZero, OpenAI Five | - |

## Setup

On SageMaker, run this from the unzipped bundle (`<folder>/Training/`). On your computer, open it from the repository's `Training/` folder.

In [ ]:
import os, time, zipfile
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
bundle = os.path.join(home, "wow_sagemaker.zip")
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    if not os.path.isdir(os.path.join(home, "wow")):
        assert os.path.exists(bundle), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(bundle).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training

# Uploading a new wow_sagemaker.zip does not replace the unzipped folder: check which version runs here
running = open("BUNDLE_VERSION").read().strip() if os.path.exists("BUNDLE_VERSION") else "an older one"
if ON_SAGEMAKER and os.path.exists(bundle):
    with zipfile.ZipFile(bundle) as z:
        uploaded = z.read("Training/BUNDLE_VERSION").decode().strip() if "Training/BUNDLE_VERSION" in z.namelist() else "unknown"
    if uploaded != running:
        print(f"*** {bundle} is version {uploaded}, but this notebook runs version {running}. Unzip it into a new folder "
              f"and open this notebook from there. ***")
print(os.getcwd(), "| bundle", running, "| glibc", os.confstr("CS_GNU_LIBC_VERSION"))

In [ ]:
# install only what is missing (PyTorch as the CPU build)
import importlib.util
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd

from simple_mappo import (NavalEnv, MAPPO, Monitor, CurriculumCallback, SaveOnIntervalCallback, BestModelCallback,
                          VideoCallback, TeamStorage, TeamSyncCallback, DEFAULT_CURRICULUM, continue_training,
                          training_status, evaluate, plot_results, plot_curriculum, plot_reward_terms, plot_progress,
                          show_videos)

## Team checkpoints

Use the **same `RUN_NAME` in every notebook** (and as your teammates): that is the one line of training the stages hand over along. Only one person can train it at a time.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates
RUN_NAME = "fleet"                   # the same in every stage notebook (and StageP)
if ON_SAGEMAKER:
    from simple_mappo.aws import team_storage_uri
    TEAM_STORAGE = team_storage_uri()     # s3://sagemaker-<region>-<account>/wow-mappo, or "s3://<team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<team-bucket>/wow-mappo"; None trains on this machine only

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
run_dir = f"runs_simple/{RUN_NAME}"
log_dir, models_dir, video_dir = (os.path.join(run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(models_dir, exist_ok=True)
if storage:
    storage.show_status()

## Configuration

The method's settings are **identical in every notebook**. They only shape a brand-new model; a continued model keeps the settings it was created with. `n_timesteps` is this session's training; rerun the notebook to continue.

In [ ]:
STAGE = 1
FORCE_STAGE = False                  # True: train this stage even if the run is on another one
KEEP_GOING = False                   # True: when this stage is passed, carry on into the next stages
                                     # in this same session instead of stopping (continuous training)

config = OrderedDict([
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles in parallel, one Unity process each
    ('decision_period', 1.0),          # game seconds between decisions
    ('sim_dt', 0.08),                  # game seconds per simulated frame (fastest stable)
    ('n_timesteps', 200_000),          # this session's training
])
mappo_settings = dict(learning_rate=3e-4, critic_learning_rate=5e-4, n_steps=256, n_minibatches=4, n_epochs=5,
                      gamma=0.995, gae_lambda=0.95, clip_range=0.2, dual_clip=3.0, ent_coef=0.01,
                      d_model=64, attn_layers=2, hidden_size=128, recurrent=True,      # the networks
                      commander=True, commander_period=10,                             # the fleet commander
                      kl_bc_coef=0.1, opponent_refresh=10)                             # imitation anchor, league
save_interval = config['n_timesteps'] // 3

pd.Series(DEFAULT_CURRICULUM[STAGE], name=f"stage {STAGE}").to_frame()

## Environment

Launches `n_envs` headless Unity players. The observation has the same size in every stage (room for 8 ships a side and 5 circles), which is what lets one network play them all.

In [ ]:
env = Monitor(NavalEnv(stages="default", start_stage=STAGE, n_envs=config['n_envs'],
                       decision_period=config['decision_period'], sim_dt=config['sim_dt'],
                       unity_log_dir=os.path.join(run_dir, "unity_logs")), log_dir)
print("battles in parallel:", env.n_envs, "| token sizes", env.dims)

## Continue the model from the previous stage

`continue_training` loads the run's latest checkpoint with all its training memory. If the run is on another stage it says which notebook to open.

In [ ]:
model = continue_training(env, STAGE, run_dir, log_dir, storage=storage, force=FORCE_STAGE, **mappo_settings)
if storage:
    storage.lock()                   # stops here if a teammate is training this run right now
model.actor                          # the captain (and the commander), as in the Week 10 notebooks

## Baselines

What the trained fleet has to beat on this stage: random legal actions, and a simple hand-written tactic. `pass_rate` is the stage's own goal (see the table at the top). Set `RUN_BASELINES = False` to skip them in later sessions.

In [ ]:
class Script:
    """Every ship sails to the first capture circle at full speed and fires at will."""
    def predict(self, obs, deterministic=False):
        E, N = obs["alive"].shape
        actions = np.zeros((E, N, len(env.head_sizes)), dtype=np.int32)
        actions[..., 0] = np.where(obs["action_mask"][..., 15] > 0.5, 15, 0)   # move option 15 = sail to circle 0
        return actions
SCRIPT_NAME = "scripted: go to the circle, fire at will"

RUN_BASELINES = True
baselines = {}
if RUN_BASELINES:
    baselines = {"random": evaluate(None, env, n_battles=24, stage=STAGE),
                 SCRIPT_NAME: evaluate(Script(), env, n_battles=24, stage=STAGE)}
    display(pd.DataFrame(baselines).T[["stage", "battles", "win_rate", "pass_rate", "mean_reward"]])

## Train

Training stops by itself as soon as the promotion requirement is met (unless `KEEP_GOING`), or after `n_timesteps`. `models/model_best.pt` always holds the checkpoint with the best rolling pass rate. The team folder gets a status update every 10,000 timesteps and a checkpoint every `save_interval`.

**Replays**: five times a session a battle is recorded and **shown right here, under the training table**, while training goes on. The files are in `runs_simple/<RUN_NAME>/videos/` (`.gif` on SageMaker, which has no ffmpeg - double-click one in the file browser; `.mp4` where ffmpeg exists), and in the team folder.

In the table: `win_rate` / `pass_rate` against the stage's own opponent.

In [ ]:
curriculum = CurriculumCallback(stop_on_promotion=not KEEP_GOING)
callbacks = [SaveOnIntervalCallback(save_interval, models_dir),     # also refreshes model_latest.pt
             curriculum,
             BestModelCallback(models_dir),
             VideoCallback(every=config['n_timesteps'] // 5, video_dir=video_dir, show=True)]   # replays appear below
if storage:
    callbacks.append(TeamSyncCallback(storage, run_dir, push_every=save_interval, status_every=10_000))

model.learn(total_timesteps=model.num_timesteps + config['n_timesteps'], callback=callbacks, log_interval=5)

## Save and hand over

Saves the model and pushes it to the team. **If you stop training early (Kernel > Interrupt), run this cell too**, so your progress is kept and the run is free for the next person.

In [ ]:
model.save(os.path.join(models_dir, "model_final"))
model.save(os.path.join(models_dir, "model_latest"))
if storage:
    storage.push(run_dir, model, status=training_status(model, run_dir))
    storage.release()

if env.stage > STAGE:
    print(f"Stage {STAGE} passed at {model.num_timesteps:,} timesteps - the model is on '{env.stages[env.stage]['name']}' now: "
          f"open the Stage{env.stage} notebook.")
elif curriculum.finished:
    print(f"The final stopping rule is met at {model.num_timesteps:,} timesteps - the curriculum is complete. "
          f"models/model_final.pt is the trained fleet (running this notebook again trains on to improve it).")
else:
    print(f"Stage {STAGE} not passed yet ({model.num_timesteps:,} timesteps) - run this notebook again to continue.")

## Progress on this stage

In [ ]:
if storage:
    storage.show_status()
plot_results(log_dir, title=f"Stage {STAGE} learning curve", stage=STAGE)
plot_curriculum(log_dir)                     # win rate per stage, one colour each
plot_reward_terms(log_dir, stage=STAGE)      # what the fleet is being paid for on this stage

In [ ]:
plot_progress(log_dir)                       # the PPO diagnostics

## Evaluation

Fresh battles on this stage against its own opponent, with sampled actions. The best checkpoint is evaluated too.

In [ ]:
results = dict(baselines)
results["MAPPO (latest)"] = evaluate(model, env, n_battles=40, stage=STAGE)
best_path = os.path.join(models_dir, "model_best.pt")
if os.path.exists(best_path):
    stage_now = env.stage
    best = MAPPO.load(best_path, env=env)          # (loading moves the environment to the stage it was saved on)
    results["MAPPO (best)"] = evaluate(best, env, n_battles=40, stage=STAGE)
    env.set_stage(stage_now)
pd.DataFrame(results).T[["battles", "win_rate", "pass_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost"]]

## Put the fleet into your Unity game

Exports the best checkpoint (captains + fleet commander) as the game's policy file. The game runs it in plain C# (`Assets/Scripts/RL/RLPolicy.cs`), no Python needed. From the repository it goes straight into `Assets/StreamingAssets/RL/naval_policy.bin` (the old file is kept as `.bak`); elsewhere (SageMaker) it goes into the run folder - download it (right-click > Download) and copy it there.

Then in Unity: **Play**, and on the setup screen set **ENEMY AI: LEARNED** (fight against it) or **YOUR FLEET: LEARNED** (watch it fight). The game reads the file when it starts, so press Play again after exporting. From a computer later: `python export_simple.py` (or `--team-storage s3://... --run-name fleet` to fetch a teammate's run first).

In [ ]:
from simple_mappo.export import export_policy
checkpoint = os.path.join(models_dir, "model_best.pt" if os.path.exists(os.path.join(models_dir, "model_best.pt")) else "model_latest.pt")
in_repo = os.path.isdir(os.path.join("..", "Assets", "StreamingAssets"))
target = os.path.join("..", "Assets", "StreamingAssets", "RL", "naval_policy.bin") if in_repo else os.path.join(run_dir, "naval_policy.bin")
print("exported", checkpoint, "->", os.path.abspath(export_policy(checkpoint, target)))

## Replays from this stage

The newest replays recorded while training on this stage. They are a 2D map drawn from the game's data (the training player has no graphics); to see the model in the real game's graphics, run `python watch_simple.py --model <model_final.pt>` on a computer with a screen.

In [ ]:
show_videos(video_dir, contains=f"_stage{STAGE}", newest=3)

In [ ]:
env.close()